# Feature Geometry: SAE Dictionary Analysis

This notebook demonstrates IRTK's `feature_geometry` module for analyzing the
population-level geometry of SAE feature dictionaries:

1. **Feature splitting** – how features split when SAE width increases
2. **Absorption detection** – one feature absorbing another's role
3. **Universality** – shared features across models
4. **Interaction graphs** – co-occurrence and suppression
5. **Decoder geometry** – pairwise cosines, near-duplicates, norms

## Why This Matters

SAE dictionary geometry analyzes the structure of learned feature dictionaries — how features split, absorb each other, or form interaction graphs. Understanding feature geometry is essential for evaluating whether SAEs find the 'true' features of a model or artifacts of the training procedure.

**Key references:**
- [Bricken et al. (2023) "Towards Monosemanticity"](https://transformer-circuits.pub/2023/monosemantic-features/index.html) — Sparse autoencoders find interpretable features
- [Cunningham et al. (2023) "Sparse Autoencoders Find Highly Interpretable Features"](https://arxiv.org/abs/2309.08600) — SAE features in larger language models

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np

from irtk import HookedTransformer, HookedTransformerConfig
from irtk.sae import SparseAutoencoder
from irtk.feature_geometry import (
    feature_splitting_analysis,
    feature_absorption_detection,
    feature_universality,
    feature_interaction_graph,
    decoder_geometry_stats,
)

In [ ]:
# Build a small model and two SAEs of different widths
cfg = HookedTransformerConfig(
    n_layers=2, d_model=16, n_ctx=32, d_head=4, n_heads=4, d_vocab=50,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)

sae_small = SparseAutoencoder(16, 32, key=jax.random.PRNGKey(0))
sae_large = SparseAutoencoder(16, 64, key=jax.random.PRNGKey(1))
seqs = [jnp.array([0, 1, 2, 3, 4, 5]), jnp.array([10, 11, 12, 13])]
print(f"Small SAE: {sae_small.n_features} features")
print(f"Large SAE: {sae_large.n_features} features")

## 1. Feature Splitting

When you scale up an SAE, some features split into multiple children.

In [ ]:
split = feature_splitting_analysis(
    sae_small, sae_large, model, seqs, "blocks.0.hook_resid_post"
)
print(f"Decoder cosines shape: {split['decoder_cosines'].shape}")
print(f"Split features: {len(split['split_features'])}")
print(f"Unsplit features: {len(split['unsplit_features'])}")
print(f"Mean split count: {split['mean_split_count']:.2f}")
if split['split_features']:
    si, matches, cos = split['split_features'][0]
    print(f"\nExample: small feature {si} -> large features {matches} (cos={cos:.3f})")

## 2. Absorption Detection

Test whether feature B absorbs feature A's role when B is active.

In [ ]:
absorb = feature_absorption_detection(
    sae_small, model, seqs, "blocks.0.hook_resid_post", feature_a=0, feature_b=1
)
print(f"Absorption score: {absorb['absorption_score']:.4f}")
print(f"A firing rate without B: {absorb['a_rate_without_b']:.4f}")
print(f"A firing rate with B:    {absorb['a_rate_with_b']:.4f}")
print(f"Suppression ratio:       {absorb['suppression_ratio']:.4f}")

## 3. Feature Interaction Graph

Build a graph of feature co-occurrence and suppression.

In [ ]:
graph = feature_interaction_graph(
    sae_small, model, seqs, "blocks.0.hook_resid_post", top_k=10
)
print(f"Co-occurrence matrix shape: {graph['co_occurrence'].shape}")
print(f"Suppression pairs: {len(graph['suppression_pairs'])}")
print(f"Feature indices analyzed: {graph['feature_indices'][:5]}...")

## 4. Decoder Geometry Stats

Analyze the geometry of the decoder dictionary matrix.

In [ ]:
geo = decoder_geometry_stats(sae_small)
print(f"Pairwise cosines shape: {geo['pairwise_cosines'].shape}")
print(f"Mean pairwise cosine: {geo['mean_pairwise_cosine']:.4f}")
print(f"Near-duplicate pairs: {geo['n_near_duplicates']}")
print(f"Feature norms (first 8): {geo['feature_norms'][:8].round(4)}")